

Este notebook implementa un pipeline completo de interpretabilidad local para el modelo DNF (Deep Neuro-Fuzzy) de detección de anomalías en un secador de granos. El modelo DNF combina dos ramas de inferencia: una rama neuro-difusa que opera sobre estadísticas agregadas de ventanas temporales, y una rama basada en LSTM que procesa las secuencias temporales crudas. El pipeline XAI tiene como objetivo explicar las predicciones del modelo a nivel individual, validar la coherencia de dichas explicaciones según el resultado de la detección, y transformar esas explicaciones en información operativa accionable mediante la detección de Puntos Críticos de Control (PCCs). El sistema permite que un técnico del proceso de secado reciba no solo una alarma de anomalía, sino también el motivo detrás de ella, las variables involucradas y una recomendación concreta.

El notebook tiene un carácter de **prueba manual e iterativa**. Su función principal es permitir al ingeniero explorar y ajustar la combinación óptima de tres variables configurables — `SUBSYSTEMS_PCC` (definición de subsistemas), `PCC_CATALOG` (catálogo de patrones de riesgo) y `MONITOR_POLICY` (política de estados del monitor) — hasta obtener una configuración que garantice una explicabilidad operativa coherente y fiable. Una vez encontrada la combinación satisfactoria, esta se debe transferir al archivo de configuración de producción `config/config.yaml`, que es la fuente utilizada por el flujo de inferencia en tiempo real. Los valores del notebook corresponden directamente con las claves `xai.pcc.subsystems`, `xai.pcc.catalog` y `xai.pcc.monitor_policy`.

**Validación por grupo de confusión — Evaluación de la interpretabilidad local**

La primera fase del pipeline consiste en verificar que las explicaciones generadas por el modelo tienen sentido y se diferencian según el tipo de resultado de la predicción. Para ello, se ejecuta el modelo sobre un conjunto de evaluación etiquetado y se separan las muestras en cuatro grupos según la matriz de confusión: verdaderos positivos (TP), verdaderos negativos (TN), falsos positivos (FP) y falsos negativos (FN). De cada grupo se selecciona un subconjunto de muestras, sobre las cuales se ejecuta el explicador completo.

El explicador procesa cada muestra individual y descompone la predicción en tres niveles. La rama neuro-difusa analiza las reglas activadas y extrae las variables  dominantes junto con un ratio de soporte difuso que mide qué tan consistente es la explicación de la rama. La rama LSTM emplea valores SHAP para identificar qué sensores y qué tramo temporal de la ventana (inicio, medio o final) han impulsado la decisión. Finalmente, ambas explicaciones se fusionan calculando las variables dominantes compartidas por cada rama. El resultado por muestra incluye además la identificación del subsistema dominante, el tramo SHAP predominante y la probabilidad de anomalía del modelo.

Una vez explicadas todas las muestras del muestreo, se agregan los resultados por grupo de confusión. Esta agregación revela patrones diferenciadores: los verdaderos positivos suelen compartir un subsistema dominante claro y un tramo temporal coherente con alta probabilidad; los falsos negativos pueden mostrar patrones reconocibles que el modelo no alcanza a clasificar como anomalía, lo que sugiere oportunidades de mejora; y los falsos positivos tienden a tener un margen al umbral de decisión pequeño, lo que indica activaciones borderline sobre perfiles ambiguos. El resultado de esta fase es un diagnóstico estadístico de cómo el modelo razona en cada tipo de caso.

**Detección de PCC — Perfilado de criticidad operativa**

La segunda fase toma los resultados individuales de la fase anterior y los transforma en perfiles operativos agrupados, que constituyen el puente entre la explicabilidad técnica y la detección práctica de PCC. Para entender este paso, es necesario distinguir dos conceptos: el perfilado y el catálogo de PCCs. El perfilado es un proceso exploratorio que descubre qué patrones emergen de las explicaciones del modelo. El catálogo de PCCs es un diccionario de conocimiento predefinido que asigna significado operativo a los patrones más relevantes identificados. Esta fase se encarga del perfilado.

La construcción de los perfiles comienza asignando a cada muestra los dos subsistemas dominantes y el tramo temporal SHAP dominante identificados por el explicador. En el razonamiento actual, el secador se descompone en cinco subsistemas: humedad (humedad del aire de escape y humedad del grano de entrada), transferencia térmica (temperatura de plenum, temperatura de escape y potencia del quemador), ventilación y presión (presión estática y velocidad del ventilador), control de descarga (frecuencia de descarga y temperatura de setpoint) y contexto operativo (temperatura y humedad ambiente). Cada variable explicativa pertenece a uno de estos subsistemas, y el explicador determina cuál o cuáles han contribuido más a la predicción. El tramo temporal SHAP dominante indica si la contribución clave se concentró al inicio, en el medio o al final de la ventana temporal observada.

El criterio de agrupación es, pues, la combinación de los dos primeros subsistemas dominantes con el tramo SHAP dominante. Por ejemplo, un perfil podría estar formado por "termico_transferencia" como primer subsistema, "descarga_control" como segundo, y "final" como tramo temporal. Esta tripla describe un patrón observable: las muestras agrupadas en ese perfil comparten que la explicación del modelo señala principalmente a los subsistemas de transferencia térmica y control de descarga, y que la evidencia decisiva se concentra al final de la ventana observada.

Para cada perfil, se calculan métricas que permiten evaluarlo como candidato a PCC: el número de casos observados, la probabilidad media de anomalía, el margen medio al umbral de decisión y las tasas de TP, TN, FP y FN. Un perfil con alta tasa de TP indica un patrón de anomalía real y consistente; uno con alta tasa de FN revela un patrón de riesgo que el modelo no detecta, lo que sugiere que debería monitorizarse de forma explícita; y uno con mezcla ambigua de TP y TN refleja un comportamiento operativo que no se discrimina bien y que podría requerir vigilancia continua. Los perfiles que comparten el mismo par de subsistemas (con independencia del orden) se fusionan en un perfil canónico para evaluar su relevancia acumulada.

El catálogo de PCCs, por su parte, es una tabla externa definida a partir del conocimiento del proceso y de los hallazgos del perfilado. Cada entrada del catálogo asocia una tripla (subsistema principal, subsistema secundario, tramo temporal) con un nombre operativo, una descripción del patrón y una recomendación técnica.

**Evaluación del monitor — Monitor en línea**

La fase final conecta el perfilado y el catálogo de PCCs con la operación real del secador mediante un sistema de monitorización. El monitor clasifica cada muestra en uno de tres estados: "Normal", "Vigilancia" o "Criticidad detectada". La decisión se toma comparando el triple formado por los dos subsistemas dominantes y el tramo temporal SHAP de cada muestra contra las entradas del catálogo de PCCs, y evaluando si se superan los umbrales definidos en la política de monitorización.

La política de monitorización establece los parámetros que separan los estados: un margen de decisión mínimo para considerar una muestra normal, un margen más ajustado para activar la criticidad, y un ratio de soporte difuso mínimo para que la coincidencia con el catálogo sea considerada fiable. Los parámetros de la política son:
- `normal_margin`: distancia mínima por debajo del umbral para asignar Normal. 
- `critical_margin`: distancia mínima por encima del umbral para declarar Criticidad detectada. 
- `min_support_catalog`: coherencia difusa mínima para confirmar la criticidad. 
- `min_subsystem_score`: puntuación mínima para que un subsistema pueda formar parte del perfil.
- `min_subsystem_variables`: número mínimo de variables del subsistema respaldadas por alguna de las ramas.

El ajuste de estos parámetros sigue un criterio de selección claro: la etiqueta "Normal" debe capturar una buena proporción de verdaderos negativos sin absorber un número excesivo de falsos negativos; la etiqueta "Vigilancia" debe recoger una proporción significativa de falsos negativos y falsos positivos, funcionando como zona intermedia de atención; y la etiqueta "Criticidad detectada" debe capturar la mayor parte de los verdaderos positivos con idealmente pocos o ningún falso positivo. Cuando una muestra coincide con un patrón del catálogo y sus indicadores de evidencia superan los umbrales de criticidad, el monitor emite el estado "Criticidad detectada" junto con el nombre del PCC y la recomendación asociada. Si coincide con un patrón pero no alcanza el umbral crítico o no coincide con un patrón pero supera el umbral, se asigna el estado "Vigilancia". Las muestras que no coinciden con ningún patrón del catálogo y superan el umbral de normalidad se mantienen en estado "Normal".

El resultado del monitor es un registro ordenado por prioridad que permite al operador identificar las situaciones más relevantes. La distribución cruzada de estados, patrones y tipos de grupo de confusión original permite validar la calidad del sistema según los criterios descritos: verificar que "Normal" concentra TN, que "Vigilancia" actúa como filtro de FN y FP, y que "Criticidad detectada" captura los TP con máxima pureza. Este es precisamente el criterio que guía el ajuste manual e iterativo de los parámetros de `MONITOR_POLICY` en el notebook.

**Relación entre las tres fases**

Las tres fases forman un flujo continuo que va del análisis individual al perfilado exploratorio y finalmente a la decisión operativa. La fase de validación por grupo de confusión genera explicaciones individuales que describen cómo razona el modelo para cada muestra, y permite verificar que esas explicaciones son coherentes y diferenciadas según el tipo de resultado. La fase de perfilado de criticidad operativa transforma esas explicaciones individuales en patrones agrupados, revelando qué combinaciones de subsistemas y tramos temporales aparecen con mayor frecuencia y qué tasa de anomalías o riesgo presentan. Estos patrones identificados sirven de base empírica para definir el catálogo de perfiles relevantes: cada entrada del catálogo responde a un perfil real observado en los datos. Finalmente, la fase de monitor en línea utiliza ese catálogo para convertir cada nueva observación en un estado operativo accionable. Cada fase alimenta a la siguiente: sin explicaciones individuales coherentes no hay perfiles fiables, sin perfiles caracterizados no es posible definir un catálogo significativo, y sin catálogo no hay monitor operativo.

**Aplicación en el notebook**

El notebook incluye una sección de aplicación práctica en la que se ejecuta el pipeline completo sobre datos reales del secador. En primer lugar, se definen los cinco subsistemas operativos del secador y se ejecuta la fase de validación por grupo de confusión sobre el dataset `data/raw/interpretability_val.csv`, que contiene 900.000 filas de datos de sensores que se transforman en 6.000 secuencias temporales. De cada grupo de confusión se muestrean 500 muestras, lo que permite un análisis estadístico robusto. 

A continuación, se ejecuta el perfilado de criticidad operativa sobre los mismos resultados y se obtienen los perfiles fusionados. 

Finalmente, se ejecuta el monitor en línea sobre un segundo dataset independiente, `data/raw/pcc_system_eval.csv`, con 300.000 filas que generan 2.000 secuencias, del que se muestrean hasta 50 casos por grupo de confusión. 

Esta sección de aplicación constituye, por tanto, el espacio de prueba manual donde el ingeniero itera sobre los tres componentes configurables — `SUBSYSTEMS_PCC`, `PCC_CATALOG` y `MONITOR_POLICY` — para encontrar la combinación que satisfaga los criterios de selección del monitor. Una vez identificada la configuración óptima en el notebook, los valores deben trasladarse manualmente a sus equivalentes en `config/config.yaml`, que es la fuente de configuración utilizada por el flujo de inferencia en producción. La correspondencia es directa: `SUBSYSTEMS_PCC` se copia a `xai.pcc.subsystems`, `PCC_CATALOG` se copia a `xai.pcc.catalog` y `MONITOR_POLICY` se copia a `xai.pcc.monitor_policy`. De este modo, las decisiones tomadas durante la fase de pruebas manuales se materializan en la configuración de producción, garantizando que la explicabilidad validada en el notebook se replique en la inferencia real del sistema.

# Librerí­as

In [1]:
import os
import sys
import json
import traceback
import warnings
import logging
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.data_processing.load_data import load_raw_data
from src.data_processing.preprocess import (
    create_sequences,
    stats_windows,
    _resolve_sequence_feature_columns,
)
from src.data_processing.input_validation import (
    prepare_model_input_dataframe,
    temporal_impute_partial_nulls,
    validate_model_input_data,
)
from src.predict.xai_predictor import (
    load_model_artifacts,
    _resolve_background_windows,
    _resolve_xai_runtime_config,
)
from src.xai import DNFLExplainer
from src.utils.common import load_config
from src.utils.logging import get_logger

logger = get_logger(__name__)

warnings.filterwarnings("ignore")

# Interpretabilidad local para detección de PCCs

### Constructor

In [2]:
def build_xai_evaluation_context(
    config_path: str,
    dataset_path: str,
    subsystems_config: Optional[List[Dict[str, Any]]] = None,
    pcc_catalog: Optional[Dict[Any, Dict[str, Any]]] = None,
    monitor_policy: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Prepara una evaluación usando exactamente el pipeline de producción."""
    config = load_config(config_path)
    data_cfg = config["data_processing"]
    xai_cfg = dict(_resolve_xai_runtime_config(config))

    active_pcc_cfg = dict(xai_cfg.get("pcc_cfg", {}))

    if subsystems_config is not None:
        active_pcc_cfg["subsystems"] = subsystems_config

    if pcc_catalog is not None:
        active_pcc_cfg["catalog"] = pcc_catalog

    if monitor_policy is not None:
        active_pcc_cfg["monitor_policy"] = monitor_policy

    xai_cfg["pcc_cfg"] = active_pcc_cfg

    df_input = load_raw_data(dataset_path)
    df_input.columns = (
        df_input.columns.astype(str).str.strip().str.lower()
    )

    validation_report = validate_model_input_data(
        df=df_input,
        config=config,
        context="calibracion XAI/notebook",
        require_target=False,
    )
    df_input = prepare_model_input_dataframe(
        df=df_input,
        config=config,
        context="calibracion XAI/notebook",
    )

    target_column = str(data_cfg["target_column"]).lower()
    id_column = data_cfg.get("id_column")
    id_column = str(id_column).lower() if id_column else None
    timestamp_column = str(
        data_cfg.get("timestamp_column", "timestamp")
    ).lower()
    solapamiento_beta = float(data_cfg["solapamiento_beta"])

    if target_column not in df_input.columns:
        raise ValueError(
            f"El dataset de evaluación debe contener {target_column!r} "
            "para construir TP/TN/FP/FN."
        )

    df_input = temporal_impute_partial_nulls(
        df=df_input,
        partial_null_stats=validation_report.get(
            "partial_null_stats",
            {},
        ),
        id_column=id_column,
        timestamp_column=timestamp_column,
    )

    x_sequences, y_sequences, _, _ = create_sequences(
        df=df_input,
        target_column=target_column,
        seq_length=int(data_cfg["sequence_length"]),
        solapamiento_beta=solapamiento_beta,
        id_column=id_column,
        timestamp_column=timestamp_column,
        normal_tokens=data_cfg.get("normal_tokens"),
    )

    if len(x_sequences) == 0:
        raise ValueError(
            "No se generaron secuencias para evaluación XAI."
        )

    logger.info(
        "Ventanas totales: %d, con %d registros cada una y un solapamiento de %s (%d registros).",
        len(x_sequences),
        x_sequences.shape[1],
        solapamiento_beta,
        solapamiento_beta * x_sequences.shape[1],
    )
    
    feature_names_original = _resolve_sequence_feature_columns(
        df=df_input,
        target_column=target_column,
        timestamp_column=timestamp_column,
        id_column=id_column,
    )

    stats_creation = (
        data_cfg.get("fuzzy_processing", {})
        .get("stats_creation")
    )
    if not stats_creation:
        raise ValueError(
            "data_processing.fuzzy_processing.stats_creation "
            "es obligatorio."
        )

    df_stats = stats_windows(
        x_sequences,
        feature_names=feature_names_original,
        stats_creation=stats_creation,
    )

    model, _, scalers, threshold = load_model_artifacts(config)
    scaler_x = scalers["scaler_x"]
    scaler_num = scalers["scaler_num"]

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model = model.to(device)
    model.eval()

    n_features = x_sequences.shape[-1]
    x_scaled = scaler_x.transform(
        x_sequences.reshape(-1, n_features)
    ).reshape(x_sequences.shape)

    stats_scaled = scaler_num.transform(df_stats)
    stats_scaled_df = pd.DataFrame(
        stats_scaled,
        columns=df_stats.columns,
    )

    background_windows = _resolve_background_windows(
        scaler_x=scaler_x,
        data_cfg=data_cfg,
        config=config,
    )

    explainer = DNFLExplainer(
        model=model,
        feature_names_stats=df_stats.columns.tolist(),
        feature_names_original=feature_names_original,
        pcc_cfg=xai_cfg["pcc_cfg"],
        stats_creation=stats_creation,
        model_cfg=config.get("model", {}),
    )

    return {
        "config": config,
        "data_cfg": data_cfg,
        "xai_cfg": xai_cfg,
        "model": model,
        "device": device,
        "threshold": float(threshold),
        "explainer": explainer,
        "background_windows": background_windows,
        "X": x_scaled,
        "y": np.asarray(y_sequences, dtype=int),
        "stats": stats_scaled_df,
        "feature_names_original": feature_names_original,
        "feature_names_stats": df_stats.columns.tolist(),
    }

### Validación de PCC según TP/TN/FP/FN

In [3]:
# =========================================================
# Validación por tipo de grupo (TP/TN/FP/FN)
# =========================================================

def _resolve_stats_df(split_data_df_stats: Any) -> pd.DataFrame:
    if hasattr(split_data_df_stats, "iloc"):
        return split_data_df_stats

    if isinstance(split_data_df_stats, dict):
        stats_df = split_data_df_stats.get("stats")
        if hasattr(stats_df, "iloc"):
            return stats_df

    raise TypeError(
        "split_data_df_stats debe ser un DataFrame "
        "o un diccionario con la clave 'stats'."
    )

# ---------------------------------------------------------
# Predicción final del modelo 
# ---------------------------------------------------------
def predict_model_on_split(
    model,
    split_data: Dict[str, Any],
    split_data_df_stats: Dict[str, Any],
    threshold: float = 0.5,
    batch_size: int = 128,
) -> Dict[str, Any]:
    model.eval()
    device = next(model.parameters()).device

    X = np.asarray(split_data["X"])
    y_raw = split_data.get("y", None)
    y = None if y_raw is None else np.asarray(y_raw).reshape(-1)
    S = _resolve_stats_df(split_data_df_stats)

    if len(S) != len(X):
        raise ValueError(f"Desajuste len(stats)={len(S)} vs len(X)={len(X)}.")
    if y is not None and len(y) != len(X):
        raise ValueError(f"Desajuste len(y)={len(y)} vs len(X)={len(X)}.")

    n = len(X)
    probs, scores = [], []
    probs_dl, scores_dl = [], []
    probs_fuzzy, scores_fuzzy = [], []

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)

        x_batch = X[start:end]
        s_batch = S.iloc[start:end] if hasattr(S, "iloc") else S[start:end]

        x_tensor = torch.tensor(np.asarray(x_batch), dtype=torch.float32, device=device)
        s_tensor = torch.tensor(np.asarray(s_batch), dtype=torch.float32, device=device)

        with torch.no_grad():
            out = model(x_tensor, s_stats=s_tensor)

        batch_scores = out["anomaly_score"].detach().cpu().numpy().reshape(-1)
        batch_probs = 1.0 / (1.0 + np.exp(-batch_scores))

        logit_dl = out.get("logit_anomaly_dl", out["anomaly_score"]).detach().cpu().numpy().reshape(-1)
        logit_fuzzy = out.get("logit_anomaly_fuzzy", out["anomaly_score"]).detach().cpu().numpy().reshape(-1)
        batch_probs_dl = 1.0 / (1.0 + np.exp(-logit_dl))
        batch_probs_fuzzy = 1.0 / (1.0 + np.exp(-logit_fuzzy))

        scores.extend(batch_scores.tolist())
        probs.extend(batch_probs.tolist())
        scores_dl.extend(logit_dl.tolist())
        probs_dl.extend(batch_probs_dl.tolist())
        scores_fuzzy.extend(logit_fuzzy.tolist())
        probs_fuzzy.extend(batch_probs_fuzzy.tolist())

    probs = np.asarray(probs)
    scores = np.asarray(scores)
    probs_dl = np.asarray(probs_dl)
    scores_dl = np.asarray(scores_dl)
    probs_fuzzy = np.asarray(probs_fuzzy)
    scores_fuzzy = np.asarray(scores_fuzzy)

    preds = (probs >= threshold).astype(int)
    preds_dl = (probs_dl >= threshold).astype(int)
    preds_fuzzy = (probs_fuzzy >= threshold).astype(int)

    return {
        "y_true": None if y is None else y,
        "y_pred": preds,
        "prob": probs,
        "score": scores,
        "prob_dl": probs_dl,
        "score_dl": scores_dl,
        "y_pred_dl": preds_dl,
        "prob_fuzzy": probs_fuzzy,
        "score_fuzzy": scores_fuzzy,
        "y_pred_fuzzy": preds_fuzzy,
    }

# ---------------------------------------------------------
# Índices por tipo de caso
# ---------------------------------------------------------
def build_confusion_groups(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, List[int]]:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    groups = {
        "TP": np.where((y_true == 1) & (y_pred == 1))[0].tolist(),
        "TN": np.where((y_true == 0) & (y_pred == 0))[0].tolist(),
        "FP": np.where((y_true == 0) & (y_pred == 1))[0].tolist(),
        "FN": np.where((y_true == 1) & (y_pred == 0))[0].tolist(),
    }
    return groups


# ---------------------------------------------------------
# Muestreo por grupo
# ---------------------------------------------------------
def sample_group_indices(
    groups: Dict[str, List[int]],
    n_per_group: Optional[int] = 10,
    random_state: int = 42,
) -> Dict[str, List[int]]:
    rng = np.random.default_rng(random_state)
    sampled = {}

    for g, idxs in groups.items():
        if n_per_group is None or len(idxs) <= n_per_group:
            sampled[g] = list(idxs)
        else:
            chosen = rng.choice(np.asarray(idxs), size=n_per_group, replace=False)
            sampled[g] = sorted(chosen.tolist())

    return sampled

# ---------------------------------------------------------
# Interpretabilidad local por muestra
# ---------------------------------------------------------
def run_local_interpretability_pipeline(
    sample_idx: int,
    context: Dict[str, Any],
) -> Dict[str, Any]:
    sample_idx = int(sample_idx)

    X = context["X"]
    y = context["y"]
    stats_df = context["stats"]

    if not 0 <= sample_idx < len(X):
        raise IndexError(
            f"sample_idx={sample_idx} fuera de rango; n={len(X)}."
        )

    threshold = float(context["threshold"])
    xai_cfg = context["xai_cfg"]

    xai_result = context["explainer"].explain(
        x_window=X[sample_idx],
        s_stats=np.asarray(stats_df.iloc[sample_idx]),
        background_windows=context["background_windows"],
        anomaly_threshold=threshold,
        n_background=int(xai_cfg["n_background"]),
        random_state=int(
            context["config"].get("project", {}).get("seed", 42)
        ),
        top_rules=int(xai_cfg["top_rules"]),
        top_variables=int(xai_cfg["top_variables"]),
    )

    fuzzy_report = xai_result["explanation"]["fuzzy"]
    lstm_report = xai_result["explanation"]["temporal"]
    fusion_report = xai_result["explanation"]["fusion"]
    pcc_report = xai_result["pcc"]

    model_probability = float(
        xai_result["prediction"]["anomaly_probability"]
    )
    model_class = int(model_probability >= threshold)
    true_class = int(y[sample_idx])

    group_map = {
        (1, 1): "TP",
        (0, 0): "TN",
        (0, 1): "FP",
        (1, 0): "FN",
    }
    effective_group = group_map.get(
        (true_class, model_class),
        "UNK",
    )

    fuzzy_dom_pairs = (
        fuzzy_report.get("fuzzy_case_signature", {})
        .get("dominant_variables", [])
    )
    lstm_dom_rows = lstm_report.get("top_variables", [])
    shared_variables = (
        fusion_report.get("agreement", {})
        .get("shared_variables", [])
    )

    metrics = {
        "dominant_variables_fuzzy": [
            (str(name), float(weight))
            for name, weight in fuzzy_dom_pairs
        ],
        "dominant_variables_lstm": [
            (
                str(row.get("feature")),
                float(row.get("importance", 0.0)),
            )
            for row in lstm_dom_rows
            if row.get("feature") is not None
        ],
        "shared_variables": list(shared_variables),
        "dominant_shap_span": lstm_report.get(
            "dominant_span_shap",
            "unknown",
        ),
        "subsystem_dominance": pcc_report.get(
            "subsystem_dominance",
            {},
        ),
        "support_ratio": float(
            fuzzy_report.get("fuzzy_case_signature", {})
            .get("support_ratio", 0.0)
        ),
        "jaccard_score": float(
            fusion_report.get("agreement", {}).get("score", 0.0)
        ),
    }

    return {
        "sample_idx": sample_idx,
        "true_class": true_class,
        "model_class": model_class,
        "effective_group": effective_group,
        "decision_threshold": threshold,
        "selected_threshold": threshold,
        "model_probability": model_probability,
        "model_probability_dl": float(
            xai_result["prediction"]["temporal_probability"]
        ),
        "model_probability_fuzzy": float(
            xai_result["prediction"]["fuzzy_probability"]
        ),
        "decision_margin": abs(model_probability - threshold),
        "prediction": xai_result["prediction"],
        "metrics": metrics,
        "fuzzy_report": fuzzy_report,
        "lstm_report": lstm_report,
        "fusion_report": fusion_report,
        "pcc_report": pcc_report,
        "final_report": xai_result["final_report"],
    }

# ---------------------------------------------------------
# Agregación simple de resultados por grupo
# ---------------------------------------------------------
def summarize_batch_results(
    batch_results: List[Dict[str, Any]],
    top_k: int = 10,
) -> Dict[str, Any]:
    if not batch_results:
        return {
            "n_cases": 0,
            "mean_model_prob": 0.0,
            "std_model_prob": 0.0,
            "mean_model_prob_dl": 0.0,
            "std_model_prob_dl": 0.0,
            "mean_model_prob_fuzzy": 0.0,
            "std_model_prob_fuzzy": 0.0,
            "mean_decision_threshold": 0.0,
            "std_decision_threshold": 0.0,
            "mean_decision_margin": 0.0,
            "std_decision_margin": 0.0,
            "mean_fuzzy_prob": 0.0,
            "std_fuzzy_prob": 0.0,
            "mean_dl_prob": 0.0,
            "std_dl_prob": 0.0,
            "mean_support_ratio": 0.0,
            "std_support_ratio": 0.0,
            "dominant_fuzzy_variables": [],
            "dominant_fuzzy_variables_coverage_ratio": [],
            "dominant_lstm_variables": [],
            "dominant_lstm_variables_coverage_ratio": [],
            "dominant_shap_spans": [],
            "dominant_shap_spans_coverage_ratio": [],
            "shared_variables": [],
            "shared_variables_coverage_ratio": [],
        }

    def _update_presence(counter: Counter, names: List[str]) -> None:
        for name in names:
            counter[str(name)] += 1


    def _topk_ratio(counter_obj: Dict[str, float], k: int, n_cases: int) -> List[Tuple[str, float]]:
        denom = float(max(n_cases, 1))
        ranked = sorted(counter_obj.items(), key=lambda x: x[1], reverse=True)[:k]
        return [(name, float(val) / denom) for name, val in ranked]

    model_probs = []
    model_probs_dl = []
    model_probs_fuzzy = []
    decision_margins = []
    support_ratios = []

    fuzzy_var_counter = Counter()
    lstm_var_counter = Counter()
    shap_span_counter = Counter()
    shared_var_counter = Counter()

    for row in batch_results:
        rep_f = row["fuzzy_report"]
        rep_l = row["lstm_report"]
        rep_fu = row["fusion_report"]

        fuzzy_sig = rep_f["fuzzy_case_signature"]
        fusion_summary = rep_fu.get("fusion_summary", {})
        agreement = rep_fu.get("agreement", {})
        fusion_lstm = rep_fu.get("lstm", {})

        model_probs.append(float(row["model_probability"]))
        model_probs_dl.append(float(row.get("model_probability_dl", row["model_probability"])))
        model_probs_fuzzy.append(float(row.get("model_probability_fuzzy", row["model_probability"])))
        decision_margins.append(float(row["decision_margin"]))

        support_ratios.append(float(fuzzy_sig["support_ratio"]))

        fuzzy_var_pairs = [(str(var), float(score)) for var, score in fuzzy_sig.get("dominant_variables", [])[:5]]
        fuzzy_var_names = [name for name, _ in fuzzy_var_pairs]
        _update_presence(fuzzy_var_counter, fuzzy_var_names)

        lstm_var_pairs = [
            (str(var_row.get("feature", "unknown")), float(var_row.get("importance", 0.0)))
            for var_row in fusion_lstm.get("top_variables", [])[:5]
        ]
        lstm_var_names = [name for name, _ in lstm_var_pairs]
        _update_presence(lstm_var_counter, lstm_var_names)

        dominant_span = fusion_summary.get("lstm_dominant_span_shap", rep_l.get("dominant_span_shap", "unknown"))
        dominant_span_name = str(dominant_span)
        _update_presence(shap_span_counter, [dominant_span_name])

        shared_variables = agreement.get("shared_variables", fusion_summary.get("shared_variables", []))
        shared_var_names = [str(shared_var) for shared_var in shared_variables[:5]]
        _update_presence(shared_var_counter, shared_var_names)

    return {
        "n_cases": len(batch_results),
        "mean_model_prob": float(np.mean(model_probs)),
        "std_model_prob": float(np.std(model_probs)),
        "mean_model_prob_dl": float(np.mean(model_probs_dl)),
        "std_model_prob_dl": float(np.std(model_probs_dl)),
        "mean_model_prob_fuzzy": float(np.mean(model_probs_fuzzy)),
        "std_model_prob_fuzzy": float(np.std(model_probs_fuzzy)),
        "mean_decision_margin": float(np.mean(decision_margins)),
        "std_decision_margin": float(np.std(decision_margins)),
        "mean_support_ratio": float(np.mean(support_ratios)),
        "std_support_ratio": float(np.std(support_ratios)),
        "dominant_fuzzy_variables": fuzzy_var_counter.most_common(top_k),
        "dominant_fuzzy_variables_coverage_ratio": _topk_ratio(fuzzy_var_counter, top_k, len(batch_results)),
        "dominant_lstm_variables": lstm_var_counter.most_common(top_k),
        "dominant_lstm_variables_coverage_ratio": _topk_ratio(lstm_var_counter, top_k, len(batch_results)),
        "dominant_shap_spans": shap_span_counter.most_common(top_k),
        "dominant_shap_spans_coverage_ratio": _topk_ratio(shap_span_counter, top_k, len(batch_results)),
        "shared_variables": shared_var_counter.most_common(top_k),
        "shared_variables_coverage_ratio": _topk_ratio(shared_var_counter, top_k, len(batch_results)),
    }

# ---------------------------------------------------------
# Validación de interpretabilidad por grupo de confusión
# ---------------------------------------------------------
def validate_local_pipeline_by_confusion_group(
    config_path: str,
    dataset_path: str,
    n_per_group: Optional[int] = 5,
    subsystems_config: Optional[List[Dict[str, Any]]] = None,
    pcc_catalog: Optional[Dict[Any, Dict[str, Any]]] = None,
    monitor_policy: Optional[Dict[str, Any]] = None,
    random_state: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    context = build_xai_evaluation_context(
        config_path=config_path,
        dataset_path=dataset_path,
        subsystems_config=subsystems_config,
        pcc_catalog=pcc_catalog,
        monitor_policy=monitor_policy,
    )

    seed = (
        int(random_state)
        if random_state is not None
        else int(
            context["config"]
            .get("project", {})
            .get("seed", 42)
        )
    )

    split_data = {
        "X": context["X"],
        "y": context["y"],
    }
    split_stats = {
        "stats": context["stats"],
    }

    predictions = predict_model_on_split(
        model=context["model"],
        split_data=split_data,
        split_data_df_stats=split_stats,
        threshold=context["threshold"],
    )

    base_groups = build_confusion_groups(
        predictions["y_true"],
        predictions["y_pred"],
    )
    sampled_groups = sample_group_indices(
        base_groups,
        n_per_group=n_per_group,
        random_state=seed,
    )

    results = {
        "predictions": predictions,
        "base_groups": base_groups,
        "sampled_groups": sampled_groups,
        "group_results": {
            group: []
            for group in ("TP", "TN", "FP", "FN")
        },
        "group_summaries": {},
        "decision_threshold": float(context["threshold"]),
        "context": context,
    }

    for sampled_group, indices in sampled_groups.items():
        if verbose:
            print(
                f"Procesando {sampled_group}: "
                f"{len(indices)} muestras"
            )

        for sample_idx in indices:
            row = run_local_interpretability_pipeline(
                sample_idx=sample_idx,
                context=context,
            )
            row["sampled_from_group"] = sampled_group

            results["group_results"][sampled_group].append(row)

    results["evaluated_groups"] = {
        group: [
            row["sample_idx"]
            for row in rows
        ]
        for group, rows in results["group_results"].items()
    }

    for group, rows in results["group_results"].items():
        results["group_summaries"][group] = (
            summarize_batch_results(rows)
        )

    return results


# ---------------------------------------------------------
# Presentación por consola
# ---------------------------------------------------------

def print_local_pipeline_group_summary(
    validation_report: Dict[str, Any],
    top_k: int = 5,
) -> None:
    print("\n" + "=" * 100)
    print("VALIDACIÓN DE INTERPRETABILIDAD POR GRUPOS TP / TN / FP / FN")
    print("=" * 100)

    print(f"Threshold base de muestreo: {validation_report['decision_threshold']:.4f}")

    base_groups = validation_report.get("base_groups", {})
    print("\nTamaño total de grupos base:")
    for g in ["TP", "TN", "FP", "FN"]:
        print(f"  - {g}: {len(base_groups.get(g, []))}")

    for g in ["TP", "TN", "FP", "FN"]:
        rows = validation_report["group_results"].get(g, [])
        s = validation_report["group_summaries"].get(g, {})
        if not rows:
            continue

        print("\n" + "-" * 100)
        print(f"GRUPO {g}  |  n={s.get('n_cases', 0)}")
        print("-" * 100)

        print(f"Prob. final media:    {s.get('mean_model_prob', 0):.4f} ± {s.get('std_model_prob', 0):.4f}")
        print(f"Margen al umbral:    {s.get('mean_decision_margin', 0):.4f} ± {s.get('std_decision_margin', 0):.4f}")
        print(f"Prob. rama neuro-difusa media:  {s.get('mean_model_prob_fuzzy', 0):.4f} ± {s.get('std_model_prob_fuzzy', 0):.4f}")
        print(f"Prob. rama DL media:     {s.get('mean_model_prob_dl', 0):.4f} ± {s.get('std_model_prob_dl', 0):.4f}")
        print(f"Support ratio medio:    {s.get('mean_support_ratio', 0):.4f} ± {s.get('std_support_ratio', 0):.4f}")

        sub_counts: Dict[str, int] = {}
        sub_scores_agg: Dict[str, List[float]] = {}
        for row in rows:
            sub_info = row.get("metrics", {}).get("subsystem_dominance", {})
            dom = sub_info.get("dominant_subsystem", "unknown")
            score = float(sub_info.get("dominant_score", 0.0))
            sub_counts[dom] = sub_counts.get(dom, 0) + 1
            sub_scores_agg.setdefault(dom, []).append(score)

        if sub_counts:
            print("Subsistema dominante (frecuencia | proporción):")
            for sub, cnt in sorted(sub_counts.items(), key=lambda x: -x[1]):
                print(f"  - {sub}: {cnt} casos | {cnt/len(rows):.3f}")

        print("Tramo dominante SHAP:")
        print("  Cobertura normalizada (conteo/n):")
        for name, r in s.get("dominant_shap_spans_coverage_ratio", [])[:top_k]:
            print(f"    - {name}: {r:.4f}")

        all_ranked: Dict[str, List[float]] = {}
        for row in rows:
            ranked = (
                row.get("metrics", {})
                .get("subsystem_dominance", {})
                .get("ranked_subsystems", [])
            )
            for entry in ranked:
                sub_n = str(entry.get("subsystem", "unknown"))
                all_ranked.setdefault(sub_n, []).append(float(entry.get("score", 0.0)))

### Detección de PCC

In [4]:
# =========================================================
# Perfilado de criticidad operativa 
# =========================================================

def _safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return float(default)


def _normalize_pair_list(raw_value):
    """Normaliza listas tipo [(name, weight), ...] a lista de tuplas limpias."""
    out = []
    if raw_value is None:
        return out
    if not isinstance(raw_value, (list, tuple)):
        return out
    for item in raw_value:
        if isinstance(item, (list, tuple)) and len(item) >= 1:
            name = str(item[0])
            weight = _safe_float(item[1], default=np.nan) if len(item) > 1 else np.nan
            out.append((name, weight))
        elif isinstance(item, dict):
            name = str(item.get("feature", item.get("name", "unknown")))
            weight = _safe_float(item.get("importance", item.get("weight", np.nan)), default=np.nan)
            out.append((name, weight))
    return out


def build_case_table(validation_report: dict) -> pd.DataFrame:
    """Tabla plana por muestra con top-1 y top-2 subsistemas para profiling."""
    group_results = validation_report.get("group_results", {}) or {}
    rows = []

    for sampled_group, cases in group_results.items():
        for row in cases:
            metrics = row.get("metrics", {}) or {}
            subsystem_info = metrics.get("subsystem_dominance", {}) or {}
            ranked_subsystems = subsystem_info.get("ranked_subsystems", []) or []

            top1 = str(ranked_subsystems[0].get("subsystem", "unknown")) if len(ranked_subsystems) >= 1 else "unknown"
            top2 = str(ranked_subsystems[1].get("subsystem", "none")) if len(ranked_subsystems) >= 2 else "none"
            top1_score = _safe_float(ranked_subsystems[0].get("score")) if len(ranked_subsystems) >= 1 else np.nan
            top2_score = _safe_float(ranked_subsystems[1].get("score")) if len(ranked_subsystems) >= 2 else np.nan

            fuzzy_dom = _normalize_pair_list(metrics.get("dominant_variables_fuzzy", []))
            lstm_dom = _normalize_pair_list(metrics.get("dominant_variables_lstm", []))
            shared_vars = [str(v) for v in (metrics.get("shared_variables", []) or [])]

            rows.append({
                "sample_idx": int(row.get("sample_idx", -1)),
                "sampled_from_group": str(row.get("sampled_from_group", sampled_group)),
                "effective_group": str(row.get("effective_group", "UNK")),
                "true_class": int(row.get("true_class", np.nan)),
                "model_class": int(row.get("model_class", np.nan)),
                "model_probability": _safe_float(row.get("model_probability")),
                "decision_margin": _safe_float(row.get("decision_margin")),
                "support_ratio": _safe_float(metrics.get("support_ratio")),
                "agreement_score": _safe_float(metrics.get("jaccard_score")),
                "dominant_subsystem": top1,
                "dominant_subsystem_score": _safe_float(subsystem_info.get("dominant_score", top1_score)),
                "top1_subsystem": top1,
                "top2_subsystem": top2,
                "top1_subsystem_score": top1_score,
                "top2_subsystem_score": top2_score,
                "dominant_shap_span": str(metrics.get("dominant_shap_span", "unknown")),
                "dominant_variables_fuzzy": fuzzy_dom,
                "dominant_variables_lstm": lstm_dom,
                "shared_variables": shared_vars,
            })

    df_cases = pd.DataFrame(rows)
    if df_cases.empty:
        return df_cases

    df_cases["is_fp"] = (df_cases["true_class"] == 0) & (df_cases["model_class"] == 1)
    df_cases["is_fn"] = (df_cases["true_class"] == 1) & (df_cases["model_class"] == 0)
    df_cases["is_tp"] = (df_cases["true_class"] == 1) & (df_cases["model_class"] == 1)
    df_cases["is_tn"] = (df_cases["true_class"] == 0) & (df_cases["model_class"] == 0)
    return df_cases


def summarize_profile_variable_dominance(
    df_cases: pd.DataFrame,
    branch: str = "fuzzy",
    group_cols=("top1_subsystem", "top2_subsystem", "dominant_shap_span"),
    top_k: int = 8,
) -> pd.DataFrame:
    """Dominancia de variables por perfil para fuzzy/lstm/shared."""
    if df_cases.empty:
        return pd.DataFrame()

    if branch not in {"fuzzy", "lstm", "shared"}:
        raise ValueError("branch debe ser 'fuzzy', 'lstm' o 'shared'.")

    records = []
    for group_key, gdf in df_cases.groupby(list(group_cols), dropna=False):
        freq = Counter()
        weight_sum = Counter()

        if branch in {"fuzzy", "lstm"}:
            col = f"dominant_variables_{branch}"
            for vals in gdf[col]:
                vals = vals or []
                for idx, item in enumerate(vals):
                    name = str(item[0])
                    w = _safe_float(item[1], default=np.nan)
                    if np.isnan(w):
                        w = 1.0 / (idx + 1)
                    freq[name] += 1
                    weight_sum[name] += float(w)
        else:
            for vals in gdf["shared_variables"]:
                vals = vals or []
                for idx, name in enumerate(vals):
                    name = str(name)
                    freq[name] += 1
                    weight_sum[name] += 1.0 / (idx + 1)

        n = int(len(gdf))
        key_values = group_key if isinstance(group_key, tuple) else (group_key,)
        for name, cnt in freq.most_common(top_k):
            row = {
                "branch": branch,
                "variable": name,
                "count": int(cnt),
                "coverage_ratio": float(cnt / max(n, 1)),
                "mean_weight": float(weight_sum[name] / max(cnt, 1)),
                "n_cases_profile": n,
            }
            for i, col_name in enumerate(group_cols):
                row[col_name] = key_values[i] if i < len(key_values) else "unknown"
            records.append(row)

    if not records:
        return pd.DataFrame()

    dom = pd.DataFrame(records)
    dom = dom.sort_values(
        by=[*group_cols, "coverage_ratio", "mean_weight"],
        ascending=[True] * len(group_cols) + [False, False],
    ).reset_index(drop=True)
    return dom

def _is_valid_subsystem_name(value):
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    return str(value).strip().lower() not in {"", "none", "unknown", "nan", "unk"}


def _canonical_subsystem_pair(sub1, sub2):
    if not _is_valid_subsystem_name(sub1) or not _is_valid_subsystem_name(sub2):
        return None

    sub1 = str(sub1)
    sub2 = str(sub2)

    if sub1 == sub2:
        return None

    a, b = sorted([sub1, sub2])
    return f"{a} - {b}"


def _flatten_unique(values):
    out = []
    seen = set()

    for value in values:
        if value is None:
            continue
        if isinstance(value, float) and np.isnan(value):
            continue

        items = value if isinstance(value, (list, tuple, set)) else [value]
        for item in items:
            item = str(item)
            if item not in seen:
                seen.add(item)
                out.append(item)

    return out


def summarize_merged_subsystem_pair_profiles(
    profile_summary: pd.DataFrame,
    min_cases: int = 10,
) -> pd.DataFrame:
    """Fusiona perfiles por pares observados de subsistemas, tratando A-B igual que B-A."""
    if profile_summary.empty:
        return pd.DataFrame()

    df = profile_summary.copy()

    df["dominant_subsystem_pair"] = df.apply(
        lambda row: _canonical_subsystem_pair(
            row["top1_subsystem"],
            row["top2_subsystem"],
        ),
        axis=1,
    )

    df = df[df["dominant_subsystem_pair"].notna()].copy()
    if df.empty:
        return pd.DataFrame()

    def weighted_mean(g, col):
        values = pd.to_numeric(g[col], errors="coerce")
        weights = pd.to_numeric(g["n_cases"], errors="coerce").fillna(0)
        mask = values.notna() & (weights > 0)

        if not mask.any():
            return np.nan

        return float(np.average(values[mask], weights=weights[mask]))

    records = []

    for (pair, span), g in df.groupby(
        ["dominant_subsystem_pair", "dominant_shap_span"],
        dropna=False,
    ):
        n_cases = int(g["n_cases"].sum())

        tp_count = int(g["tp_count"].sum())
        tn_count = int(g["tn_count"].sum())
        fp_count = int(g["fp_count"].sum())
        fn_count = int(g["fn_count"].sum())

        records.append({
            "dominant_subsystem_pair": pair,
            "dominant_shap_span": span,
            "n_cases": n_cases,
            "prob_mean": weighted_mean(g, "prob_mean"),
            "margin_mean": weighted_mean(g, "margin_mean"),
            "support_ratio": weighted_mean(g, "support_ratio"),
            "tp_count": tp_count,
            "tn_count": tn_count,
            "fp_count": fp_count,
            "fn_count": fn_count,
            "tp_rate": tp_count / max(n_cases, 1),
            "tn_rate": tn_count / max(n_cases, 1),
            "fp_rate": fp_count / max(n_cases, 1),
            "fn_rate": fn_count / max(n_cases, 1),
            "top_shared_vars": _flatten_unique(g.get("top_shared_vars", [])),
        })

    merged = pd.DataFrame(records)
    merged = merged[merged["n_cases"] >= int(min_cases)].copy()

    return merged.sort_values(
    by=["dominant_subsystem_pair", "n_cases", "fn_rate", "fp_rate", "margin_mean"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)

def summarize_profiles(
    df_cases: pd.DataFrame,
    group_cols=("top1_subsystem", "top2_subsystem", "dominant_shap_span"),
) -> pd.DataFrame:
    if df_cases.empty:
        return pd.DataFrame()

    prof = (
        df_cases.groupby(list(group_cols), dropna=False)
        .agg(
            n_cases=("sample_idx", "count"),
            prob_mean=("model_probability", "mean"),
            margin_mean=("decision_margin", "mean"),
            support_ratio=("support_ratio", "mean"),
            mean_subsystem_score=("dominant_subsystem_score", "mean"),
            tp_count=("is_tp", "sum"),
            tn_count=("is_tn", "sum"),
            fp_count=("is_fp", "sum"),
            fn_count=("is_fn", "sum"),
        )
        .reset_index()
    )

    denom = prof["n_cases"].clip(lower=1)
    for group in ("tp", "tn", "fp", "fn"):
        prof[f"{group}_rate"] = prof[f"{group}_count"] / denom

    return prof.sort_values(
        ["n_cases", "fn_rate", "fp_rate", "margin_mean"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

def run_profile_pipeline(validation_report: dict) -> dict:
    """Perfiles por top1_subsystem + top2_subsystem + dominant_shap_span con TP/TN/FP/FN."""
    profile_group_cols = ("top1_subsystem", "top2_subsystem", "dominant_shap_span")

    df_cases = build_case_table(validation_report)

    profile_summary = summarize_profiles(df_cases, group_cols=profile_group_cols)
    dom_fuzzy = summarize_profile_variable_dominance(df_cases, branch="fuzzy", group_cols=profile_group_cols)
    dom_lstm = summarize_profile_variable_dominance(df_cases, branch="lstm", group_cols=profile_group_cols)
    dom_shared = summarize_profile_variable_dominance(df_cases, branch="shared", group_cols=profile_group_cols)

    if not profile_summary.empty and not dom_shared.empty:
        top_shared = (
            dom_shared.sort_values(
                by=[*profile_group_cols, "coverage_ratio", "mean_weight"],
                ascending=[True, True, True, False, False],
            )
            .groupby(list(profile_group_cols), dropna=False)["variable"]
            .apply(lambda s: list(pd.Series(s).head(3)))
            .reset_index(name="top_shared_vars")
        )
        profile_summary = profile_summary.merge(top_shared, on=list(profile_group_cols), how="left")
    else:
        profile_summary["top_shared_vars"] = [[] for _ in range(len(profile_summary))]

    merged_pair_profiles = summarize_merged_subsystem_pair_profiles(
        profile_summary,
        min_cases=10,
    )


    return {
        "df_cases": df_cases,
        "profile_summary": profile_summary,
        "merged_pair_profiles": merged_pair_profiles,
        "dominance": {
            "fuzzy": dom_fuzzy,
            "lstm": dom_lstm,
            "shared": dom_shared,
        },
}
    

def print_merged_pair_profiles(
    profiles: dict,
    min_cases: int = 10,
) -> None:
    
    profile_columns = [
    "dominant_subsystem_pair",
    "dominant_shap_span",
    "n_cases",
    "prob_mean",
    "margin_mean",
    "support_ratio",
    "fn_rate",
    "fp_rate",
    "tn_rate",
    "tp_rate",
    "tp_count",
    "tn_count",
    "fp_count",
    "fn_count",
    "top_shared_vars",
]

    merged = profiles.get("merged_pair_profiles", pd.DataFrame())

    if merged.empty:
        print("No hay perfiles fusionados para mostrar.")
        return

    df = merged.copy()

    if min_cases is not None:
        df = df[df["n_cases"] >= int(min_cases)]

    if df.empty:
        print(f"No hay perfiles fusionados con n_cases >= {min_cases}.")
        return

    cols = [col for col in profile_columns if col in df.columns]

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_columns", None,
        "display.width", 180,
    ):
        for pair, g in df.groupby("dominant_subsystem_pair", sort=True):
            pair_label = str(pair).replace(" - ", " y ")

            block = (
                g.sort_values(
                    by=["n_cases", "fn_rate", "fp_rate", "margin_mean"],
                    ascending=[False, False, False, True],
                )
                .reset_index(drop=True)
            )

            print(f"\n\nPerfiles fusionados para {pair_label}:")
            print(block[cols])

### Evaluación monitor

In [5]:
# =========================================================
# Evaluación de monitor online 
# =========================================================

def run_online_monitor_evaluation(
    config_path: str,
    dataset_path: str,
    n_per_group: Optional[int] = 6,
    subsystems_config: Optional[List[Dict[str, Any]]] = None,
    pcc_catalog: Optional[Dict[Any, Dict[str, Any]]] = None,
    monitor_policy: Optional[Dict[str, Any]] = None,
    random_state: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    validation_report = (
        validate_local_pipeline_by_confusion_group(
            config_path=config_path,
            dataset_path=dataset_path,
            n_per_group=n_per_group,
            subsystems_config=subsystems_config,
            pcc_catalog=pcc_catalog,
            monitor_policy=monitor_policy,
            random_state=random_state,
            verbose=verbose,
        )
    )

    rows = []

    for sampled_group, cases in (
        validation_report.get("group_results", {}).items()
    ):
        for case in cases:
            pcc = case["pcc_report"]

            rows.append({
                "sample_idx": int(case["sample_idx"]),
                "sampled_from_group": sampled_group,
                "effective_group": case["effective_group"],
                "true_class": int(case["true_class"]),
                "model_class": int(case["model_class"]),
                "state": pcc.get("state"),
                "pcc_candidate": pcc.get("pcc_candidate"),
                "probability": float(pcc.get("probability")),
                "margin": float(pcc.get("margin")),
                "threshold": float(pcc.get("threshold")),
                "support_ratio": float(
                    pcc.get("support_ratio", 0.0)
                ),
                "top1_subsystem": pcc.get("top1_subsystem"),
                "top2_subsystem": pcc.get("top2_subsystem"),
                "subsystem_pair": pcc.get("subsystem_pair"),
                "dominant_shap_span": pcc.get(
                    "dominant_shap_span"
                ),
                "key_variables": pcc.get("key_variables", []),
                "recommendation": pcc.get("recommendation"),
                "message": pcc.get("message"),
            })

    monitor_df = pd.DataFrame(rows)

    if not monitor_df.empty:
        state_order = {
            "Criticidad detectada": 0,
            "Vigilancia": 1,
            "Normal": 2,
        }

        monitor_df["_state_order"] = (
            monitor_df["state"]
            .map(state_order)
            .fillna(3)
        )

        monitor_df = (
            monitor_df.sort_values(
                by=[
                    "_state_order",
                    "probability",
                    "margin",
                ],
                ascending=[True, False, False],
            )
            .drop(columns="_state_order")
            .reset_index(drop=True)
        )

    return {
        "validation_report": validation_report,
        "monitor_df": monitor_df,
    }

# Aplicación

In [6]:
SUBSYSTEMS_PCC = [
    {
        "name": "humedad",
        "features": ["exhaust_air_humidity", "grain_moisture_in"],
    },
    {
        "name": "termico_transferencia",
        "features": ["plenum_temp", "exhaust_air_temp", "burner_power"],
    },
    {
        "name": "ventilacion_presion",
        "features": ["static_pressure", "fan_speed"],
    },
    {
        "name": "descarga_control",
        "features": ["discharge_frequency", "setpoint_temp"],
    },
    {
        "name": "contexto_operativo",
        "features": ["ambient_temp", "ambient_humidity"],
    }
]

In [7]:
try:
    local_validation = validate_local_pipeline_by_confusion_group(
        config_path="config/config.yaml",
        dataset_path="data/raw/interpretability_val.csv",
        n_per_group=500,
        random_state=42,
        verbose=True,
        subsystems_config=SUBSYSTEMS_PCC,
    )

    print_local_pipeline_group_summary(local_validation, top_k=5)

except Exception:
    traceback.print_exc()
    raise

2026-07-21 13:22:43. Cargando datos desde data/raw/interpretability_val.csv
2026-07-21 13:22:45. Dataset cargado: 900000 filas x 17 columnas
2026-07-21 13:22:47. Columna de identificador de ciclo 'cycle_id' detectada en calibracion XAI/notebook.
2026-07-21 13:22:49. Columna temporal 'timestamp' detectada como 'datetime' en calibracion XAI/notebook.
2026-07-21 13:22:49. Validacion completa superada en calibracion XAI/notebook: 900000 filas validas, 11 sensores requeridos presentes, timestamp obligatorio 'timestamp' valido.
2026-07-21 13:23:02. Ventanas totales: 6000, con 240 registros cada una y un solapamiento de 0.5 (120 registros).
2026-07-21 13:23:02. ==============================
2026-07-21 13:23:02. Arquitectura configurada:
2026-07-21 13:23:02. ==============================
2026-07-21 13:23:02. Rama DL: Input (240, 11) -> LSTM(hidden=64, layers=1, bidir=False) -> Atención -> Embedding 32 -> Head anom 1
2026-07-21 13:23:02. Rama Fuzzy: Input 55 -> Membership (3 por feature) -> R

In [8]:
profiles = run_profile_pipeline(
    validation_report=local_validation,
)

print_merged_pair_profiles(
    profiles,
    min_cases=10,
)



Perfiles fusionados para contexto_operativo y descarga_control:
                 dominant_subsystem_pair dominant_shap_span  n_cases  prob_mean  margin_mean  support_ratio   fn_rate  fp_rate   tn_rate   tp_rate  tp_count  tn_count  fp_count  \
0  contexto_operativo - descarga_control              final       13   0.772901     0.223681       0.592910  0.307692      0.0  0.153846  0.538462         7         2         0   
1  contexto_operativo - descarga_control             inicio       11   0.469168     0.260832       0.293612  0.727273      0.0  0.272727  0.000000         0         3         0   

   fn_count                                            top_shared_vars  
0         4  [discharge_frequency, exhaust_air_humidity, ambient_temp]  
1         8  [discharge_frequency, exhaust_air_humidity, ambient_temp]  


Perfiles fusionados para contexto_operativo y termico_transferencia:
                      dominant_subsystem_pair dominant_shap_span  n_cases  prob_mean  margin_mean  supp

In [9]:
MONITOR_POLICY = {
    "normal_margin": 0.35,
    "critical_margin": 0.15,
    "min_support_catalog": 0.75,
    "min_subsystem_score": 0.05,
    "min_subsystem_variables": 1,
}

PCC_CATALOG = {

    # ==========================================================
    # PCC CANDIDATOS
    # ==========================================================

    ("humedad", "termico_transferencia", "final"): {
        "name": "PCC térmico-humedad tardío",
        "message": (
            "Perfil crítico asociado al acoplamiento entre transferencia térmica "
            "y eliminación de humedad."
        ),
        "recommendation": (
            "Revisar evolución de humedad del grano, temperatura de plenum, temperatura de salida "
            "y potencia del quemador. Validar que la transferencia térmica permite una evacuación adecuada de humedad."
        ),
    },

    ("descarga_control", "termico_transferencia", "medio"): {
        "name": "PCC térmico-descarga intermedio",
        "message": (
            "Perfil altamente discriminativo asociado a la interacción entre "
            "transferencia térmica y dinámica de descarga del material."
        ),
        "recommendation": (
            "Comprobar que la regulación de descarga mantiene tiempos de residencia "
            "adecuados y no compromete la transferencia térmica ni la uniformidad "
            "del secado."
        ),
    },

    # ==========================================================
    # PERFILES DE VIGILANCIA
    # ==========================================================

    ("descarga_control", "termico_transferencia", "final"): {
        "name": "Perfil ambiguo térmico-descarga tardío",
        "message": (
            "Perfil frecuente asociado a la interacción entre transferencia térmica "
            "y control de descarga, con comportamiento operativo ambiguo."
        ),
        "recommendation": (
            "Mantener seguimiento reforzado de la frecuencia de descarga, "
            "temperaturas características del proceso y potencia térmica aplicada. "
        ),
    },

    ("descarga_control", "ventilacion_presion", "final"): {
        "name": "Perfil ambiguo descarga-ventilación tardío",
        "message": (
            "Perfil asociado a posibles casos anómalos sobre la interacción entre descarga del material y "
            "condiciones de ventilación/presión."
        ),
        "recommendation": (
            "Monitorizar conjuntamente frecuencia de descarga, presión estática "
            "y condiciones de ventilación."
        ),
    },
}

In [10]:
online_out = run_online_monitor_evaluation(
    config_path="config/config.yaml",
    dataset_path="data/raw/pcc_system_eval.csv",
    n_per_group=50,
    random_state=42,
    verbose=True,
    subsystems_config=SUBSYSTEMS_PCC,
    pcc_catalog=PCC_CATALOG,
    monitor_policy=MONITOR_POLICY,
)

display(online_out["monitor_df"]["state"].value_counts(dropna=False))

2026-07-21 13:40:50. Cargando datos desde data/raw/pcc_system_eval.csv
2026-07-21 13:40:50. Dataset cargado: 300000 filas x 17 columnas
2026-07-21 13:40:51. Columna de identificador de ciclo 'cycle_id' detectada en calibracion XAI/notebook.
2026-07-21 13:40:52. Columna temporal 'timestamp' detectada como 'datetime' en calibracion XAI/notebook.
2026-07-21 13:40:52. Validacion completa superada en calibracion XAI/notebook: 300000 filas validas, 11 sensores requeridos presentes, timestamp obligatorio 'timestamp' valido.
2026-07-21 13:40:56. Ventanas totales: 2000, con 240 registros cada una y un solapamiento de 0.5 (120 registros).
2026-07-21 13:40:56. ==============================
2026-07-21 13:40:56. Arquitectura configurada:
2026-07-21 13:40:56. ==============================
2026-07-21 13:40:56. Rama DL: Input (240, 11) -> LSTM(hidden=64, layers=1, bidir=False) -> Atención -> Embedding 32 -> Head anom 1
2026-07-21 13:40:56. Rama Fuzzy: Input 55 -> Membership (3 por feature) -> Reglas

state
Vigilancia              103
Criticidad detectada     31
Normal                   29
Name: count, dtype: int64

In [11]:
online_out['monitor_df'].value_counts(['state', 'effective_group'], dropna=False)

state                 effective_group
Vigilancia            TN                 37
                      FN                 34
Criticidad detectada  TP                 29
Vigilancia            TP                 21
Normal                FN                 16
                      TN                 13
Vigilancia            FP                 11
Criticidad detectada  FP                  2
Name: count, dtype: int64

In [12]:
online_out['monitor_df'].value_counts(['state', 'pcc_candidate', 'effective_group'], dropna=False).sort_index(level=[0, 1, 2], ascending=[True, True, True])

state                 pcc_candidate                               effective_group
Criticidad detectada  PCC térmico-descarga intermedio             TP                  3
                      PCC térmico-humedad tardío                  FP                  2
                                                                  TP                  4
                      Perfil ambiguo descarga-ventilación tardío  TP                  1
                      Perfil ambiguo térmico-descarga tardío      TP                 21
Normal                No identificado                             FN                 16
                                                                  TN                 13
Vigilancia            No identificado                             FN                 18
                                                                  FP                  5
                                                                  TN                  5
                                      